# Event Distribution

This notebook serves to calculate the distribution of event totals based on the statistical info summary files from real UFDRs (output of utils/statistics/summary.py script).

In [1]:
import json
from pathlib import Path

import pandas as pd
from scipy import interpolate
import numpy as np

In [2]:
EVENT_TYPES = {'Chat','Contact','Image','Video','Audio','Call','Location','Cookie','Password','WebBookmark','VisitedPage','Email','InstalledApplication','Document','Text','Notification','Journey','Note','CarvedString'}
EVENT_TYPES |= {'CalendarEntry', 'SMS', 'InstantMessage'}

In [3]:
info_dirpath = Path('/mnt/CTO/Users/DanielHurwitz/Titan/statistics/results/')

In [10]:
def get_info(info_path):
    with info_path.open() as f:
        info_json = json.load(f)
    info_counts = {'ufdr': info_path.name[4:-5], 'report_id': info_json.get('report_id'), 'report_version': info_json.get('report_version')}
    for model_type, infos in info_json['model_type_to_infos'].items():
        if model_type in EVENT_TYPES:
            if model_type == 'Chat':
                info_count = sum(len(info['messages']) for info in infos)
            else:
                info_count = len(infos)
            # if model_type == 'SMS':
            #     model_type = 'InstantMessage'   # SMS is from older PA versions
            # if f'Event.{model_type}' in info_counts:
            #     raise Exception(f'Event.{model_type} model type already exist in info_counts dictionary!')
            info_counts[f'Event.{model_type}'] = info_count
    for file_info in info_json['file_infos']:
        if file_info["category"] in EVENT_TYPES:
            category = f'File.{file_info["category"]}'
            info_counts[category] = info_counts.get(category, 0) + 1
    return info_counts

In [11]:
df_info = pd.DataFrame(map(get_info, info_dirpath.iterdir()))
df_info.set_index(['ufdr', 'report_id', 'report_version'], inplace=True)
df_info = df_info.reindex(sorted(df_info.columns), axis=1)
df_info.sort_values('report_version', ascending=False)

,,,Event.CalendarEntry,Event.Call,Event.CarvedString,Event.Chat,Event.Contact,Event.Cookie,Event.Email,Event.InstalledApplication,Event.InstantMessage,Event.Journey,...,Event.Notification,Event.Password,Event.SMS,Event.VisitedPage,Event.WebBookmark,File.Audio,File.Document,File.Image,File.Text,File.Video
ufdr,report_id,report_version,,,,,,,,,,,,,,,,,,,,,
18,a3012bd5-1395-4313-a5db-7458850d6330,7.48,127.0,454.0,NaN,18558.0,913.0,454.0,197.0,694.0,594.0,2.0,...,NaN,NaN,NaN,7928.0,6.0,5270.0,200.0,32416.0,NaN,1005.0
13,473a6d67-1635-4d45-a851-41de0a93edba,7.48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
101,be70091b-acb3-409a-8694-f74154f5cae8,7.48,127.0,798.0,NaN,8261.0,7756.0,362.0,NaN,644.0,104.0,22.0,...,NaN,700.0,NaN,16085.0,7.0,22668.0,151.0,48560.0,18037.0,4054.0
100,be70091b-acb3-409a-8694-f74154f5cae8,7.48,127.0,798.0,NaN,8261.0,7756.0,362.0,NaN,644.0,104.0,22.0,...,NaN,700.0,NaN,16085.0,7.0,22668.0,151.0,48560.0,18037.0,4054.0
10,27c910e4-85ca-492c-898c-a945b5a28318,7.48,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,de7df5c9-ddf8-4179-b2da-dad57a1a35ed,3.0,NaN,22.0,NaN,838.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61,955ca703-4fac-4f1a-8558-ca9547bc7e0d,3.0,988.0,27.0,NaN,8053.0,942.0,248.0,4079.0,78.0,NaN,NaN,...,NaN,NaN,3003.0,NaN,NaN,NaN,NaN,13747.0,442.0,NaN
60,6a235eb5-6508-4327-aa8c-94b78eb30413,3.0,2358.0,133.0,1070.0,8213.0,224.0,1966.0,3208.0,80.0,NaN,NaN,...,NaN,51.0,3594.0,766.0,29.0,547.0,NaN,45490.0,1362.0,12.0


In [12]:
df_info.columns

Index(['Event.CalendarEntry', 'Event.Call', 'Event.CarvedString', 'Event.Chat',
       'Event.Contact', 'Event.Cookie', 'Event.Email',
       'Event.InstalledApplication', 'Event.InstantMessage', 'Event.Journey',
       'Event.Location', 'Event.Note', 'Event.Notification', 'Event.Password',
       'Event.SMS', 'Event.VisitedPage', 'Event.WebBookmark', 'File.Audio',
       'File.Document', 'File.Image', 'File.Text', 'File.Video'],
      dtype='object')

In [13]:
df_info

,,,Event.CalendarEntry,Event.Call,Event.CarvedString,Event.Chat,Event.Contact,Event.Cookie,Event.Email,Event.InstalledApplication,Event.InstantMessage,Event.Journey,...,Event.Notification,Event.Password,Event.SMS,Event.VisitedPage,Event.WebBookmark,File.Audio,File.Document,File.Image,File.Text,File.Video
ufdr,report_id,report_version,,,,,,,,,,,,,,,,,,,,,
89,466ed3f8-526f-4d6f-8b6c-d0950c6baea1,7.34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27986.0,NaN,1241.0
9,d7fcbd84-8f55-4d9c-bc0d-c4eccf8d5a18,7.48,NaN,170.0,NaN,5912.0,900.0,81.0,114.0,529.0,24.0,NaN,...,3.0,330.0,NaN,4.0,7.0,1721.0,60.0,8098.0,1840.0,100.0
90,cb2c9252-268b-49cd-bdf3-c65d43a4c6bf,7.34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
91,bf672818-cfc3-47ab-9c4e-87112164c1fa,7.34,53.0,757.0,NaN,4199.0,10837.0,3379.0,1964.0,579.0,574.0,NaN,...,60.0,731.0,NaN,14898.0,4.0,828.0,29.0,65759.0,7176.0,1936.0
92,ef501551-3e8e-4804-973c-5a18c2600e00,7.34,213.0,2000.0,NaN,NaN,108.0,2.0,NaN,114.0,6999.0,NaN,...,NaN,NaN,NaN,NaN,NaN,94.0,8.0,3155.0,2388.0,61.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,1f1bf7e1-19d9-48ad-b63d-060be3793ccb,3.0.0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57,c7f1a901-d801-4278-bd33-edd8f2126410,3.0.0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58,bc9a115a-e90f-4fe4-a5ac-1778130d90b1,3.0,48.0,111.0,292.0,5132.0,751.0,1678.0,1094.0,105.0,NaN,NaN,...,NaN,58.0,1554.0,14.0,26.0,362.0,19.0,47521.0,1118.0,64.0


In [14]:
df_info.index.get_level_values('report_version').value_counts()

7.34       23
3.0.0.0    21
7.48       18
5.3.2.1    12
5.2.0.0    12
3.1.0.0     6
3.0         5
5.6         5
5.0.0.0     5
5.3.1.0     5
4.2.0.1     4
5.6.1       1
Name: report_version, dtype: int64

In [15]:
(df_info.isna().sum() / len(df_info) * 100).sort_values(ascending=False)

Event.Notification            94.067797
Event.CarvedString            91.525424
Event.Journey                 79.661017
Event.InstantMessage          73.728814
Event.SMS                     65.254237
Event.Note                    64.406780
Event.Email                   64.406780
Event.Password                63.559322
File.Document                 61.016949
Event.CalendarEntry           59.322034
Event.VisitedPage             55.084746
Event.WebBookmark             55.084746
Event.Cookie                  54.237288
Event.InstalledApplication    52.542373
File.Text                     52.542373
File.Audio                    48.305085
Event.Call                    46.610169
File.Video                    45.762712
Event.Location                42.372881
Event.Contact                 40.677966
Event.Chat                    38.135593
File.Image                    34.745763
dtype: float64

In [16]:
df_info.isna().sum(axis=1).sort_values(ascending=False)

ufdr  report_id                             report_version
7     e7b3d540-42da-4a3a-9a28-11b43ce25355  5.6               22
36    f847edd6-c2d6-429a-b65d-18b256aa75f6  3.0.0.0           22
52    172b3531-7680-4532-9e99-afab5ac208e3  5.0.0.0           22
37    2023ed38-5b6b-4fbb-b5ee-05a859e2180e  3.0.0.0           22
42    dbec95ef-455d-45aa-95c1-10e07fc43918  3.0.0.0           22
                                                              ..
58    bc9a115a-e90f-4fe4-a5ac-1778130d90b1  3.0                3
73    929c390a-3f5e-4190-bcd2-df505b995f14  5.3.2.1            2
86    95a3fb99-a10c-4fda-81f7-1ea20bd5d09a  7.34               2
72    1a001135-1a2b-495e-8489-79454eb68493  5.3.2.1            1
71    991a2ce0-7d6a-49ee-896a-150ed4224e7b  5.3.2.1            0
Length: 118, dtype: int64

In [17]:
df_info.describe()

,Event.CalendarEntry,Event.Call,Event.CarvedString,Event.Chat,Event.Contact,Event.Cookie,Event.Email,Event.InstalledApplication,Event.InstantMessage,Event.Journey,...,Event.Notification,Event.Password,Event.SMS,Event.VisitedPage,Event.WebBookmark,File.Audio,File.Document,File.Image,File.Text,File.Video
count,48.000000,63.00000,10.000000,73.000000,70.000000,54.000000,42.000000,56.000000,31.000000,24.000000,...,7.000000,43.000000,41.000000,53.000000,53.000000,61.000000,46.000000,77.000000,56.000000,64.000000
mean,310.354167,878.84127,6193.000000,21815.123288,2836.542857,1511.129630,2600.833333,324.196429,433.516129,10.666667,...,63.857143,256.418605,8711.975610,3297.754717,21.509434,2596.868852,70.739130,16167.857143,2445.714286,704.562500
std,638.804252,1515.37134,12568.943357,37469.213679,5314.413902,1684.984139,8280.616967,244.547926,1265.085317,11.032232,...,53.976626,264.509990,19329.308504,6817.425272,33.543097,5880.033332,98.923637,20865.114480,4218.974958,1439.919066
min,1.000000,2.00000,292.000000,1.000000,4.000000,2.000000,1.000000,19.000000,1.000000,1.000000,...,3.000000,1.000000,1.000000,2.000000,1.000000,2.000000,1.000000,1.000000,3.000000,1.000000
25%,46.000000,204.00000,721.250000,1340.000000,173.500000,362.250000,33.250000,95.000000,5.000000,2.000000,...,16.500000,46.500000,192.000000,60.000000,6.000000,55.000000,4.250000,972.000000,77.750000,59.250000
50%,94.000000,504.00000,1440.500000,8213.000000,906.500000,742.500000,197.000000,249.500000,74.000000,8.000000,...,60.000000,208.000000,1554.000000,341.000000,9.000000,362.000000,22.000000,8916.000000,539.000000,292.500000
75%,178.750000,766.00000,3363.000000,19889.000000,2603.500000,2180.500000,1294.500000,535.250000,373.000000,13.750000,...,109.000000,421.500000,4638.000000,2485.000000,20.000000,1910.000000,95.750000,26388.000000,2458.000000,581.000000
max,3001.000000,9485.00000,41062.000000,214753.000000,23119.000000,7064.000000,52234.000000,800.000000,6999.000000,45.000000,...,133.000000,1228.000000,86084.000000,33187.000000,184.000000,32349.000000,394.000000,90277.000000,18037.000000,8674.000000


In [18]:
df_info.sum(axis=1).sort_values(ascending=False)

ufdr  report_id                             report_version
114   47cfabea-d00c-4056-b6f4-9cf4b8464028  5.3.2.1           406478.0
44    117901e2-4a57-4973-a46d-2787e3785647  5.2.0.0           187501.0
27    b1cfa19a-9a23-48ab-92ef-6d056ba688b1  5.2.0.0           172248.0
25    4e856ef2-8350-4797-a116-8cd774fdb0db  5.2.0.0           153416.0
72    1a001135-1a2b-495e-8489-79454eb68493  5.3.2.1           142306.0
                                                                ...   
62    eaafc4ac-b11c-40a9-b10d-1fbb90024345  5.0.0.0                0.0
63    184dad8f-9be9-4aee-9f0f-e62a88be8e4f  5.0.0.0                0.0
64    f847edd6-c2d6-429a-b65d-18b256aa75f6  3.0.0.0                0.0
21    NaN                                   NaN                    0.0
65    fcf74edf-610e-4542-9a47-167bee46056a  5.0.0.0                0.0
Length: 118, dtype: float64

# Impute NaN values

In [19]:
x_indx, y_indx = np.meshgrid(np.arange(0, df_info.shape[1]), np.arange(0, df_info.shape[0]))
x_indx, y_indx

(array([[ 0,  1,  2, ..., 19, 20, 21],
        [ 0,  1,  2, ..., 19, 20, 21],
        [ 0,  1,  2, ..., 19, 20, 21],
        ...,
        [ 0,  1,  2, ..., 19, 20, 21],
        [ 0,  1,  2, ..., 19, 20, 21],
        [ 0,  1,  2, ..., 19, 20, 21]]),
 array([[  0,   0,   0, ...,   0,   0,   0],
        [  1,   1,   1, ...,   1,   1,   1],
        [  2,   2,   2, ...,   2,   2,   2],
        ...,
        [115, 115, 115, ..., 115, 115, 115],
        [116, 116, 116, ..., 116, 116, 116],
        [117, 117, 117, ..., 117, 117, 117]]))

In [20]:
grid = df_info.values
grid

array([[       nan,        nan,        nan, ..., 2.7986e+04,        nan,
        1.2410e+03],
       [       nan, 1.7000e+02,        nan, ..., 8.0980e+03, 1.8400e+03,
        1.0000e+02],
       [       nan,        nan,        nan, ..., 2.0000e+00,        nan,
               nan],
       ...,
       [4.8000e+01, 1.1100e+02, 2.9200e+02, ..., 4.7521e+04, 1.1180e+03,
        6.4000e+01],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [1.8400e+02, 4.1200e+02,        nan, ..., 2.6443e+04, 2.5600e+02,
        6.5400e+02]])

In [21]:
grid_masked = np.ma.masked_invalid(grid)
grid_masked

masked_array(
  data=[[--, --, --, ..., 27986.0, --, 1241.0],
        [--, 170.0, --, ..., 8098.0, 1840.0, 100.0],
        [--, --, --, ..., 2.0, --, --],
        ...,
        [48.0, 111.0, 292.0, ..., 47521.0, 1118.0, 64.0],
        [--, --, --, ..., --, --, --],
        [184.0, 412.0, --, ..., 26443.0, 256.0, 654.0]],
  mask=[[ True,  True,  True, ..., False,  True, False],
        [ True, False,  True, ..., False, False, False],
        [ True,  True,  True, ..., False,  True,  True],
        ...,
        [False, False, False, ..., False, False, False],
        [ True,  True,  True, ...,  True,  True,  True],
        [False, False,  True, ..., False, False, False]],
  fill_value=1e+20)

In [22]:
valid_xs = x_indx[~grid_masked.mask]
valid_ys = y_indx[~grid_masked.mask]
valid_zs = grid_masked[~grid_masked.mask]

In [23]:
zs_interp = interpolate.griddata((valid_xs, valid_ys), valid_zs.ravel(),
                                 (x_indx, y_indx), method='linear')

In [24]:
df_info_interpolated = pd.DataFrame(zs_interp, columns=df_info.columns, index=df_info.index)
df_info_interpolated

,,,Event.CalendarEntry,Event.Call,Event.CarvedString,Event.Chat,Event.Contact,Event.Cookie,Event.Email,Event.InstalledApplication,Event.InstantMessage,Event.Journey,...,Event.Notification,Event.Password,Event.SMS,Event.VisitedPage,Event.WebBookmark,File.Audio,File.Document,File.Image,File.Text,File.Video
ufdr,report_id,report_version,,,,,,,,,,,,,,,,,,,,,
89,466ed3f8-526f-4d6f-8b6c-d0950c6baea1,7.34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27986.000000,14613.500000,1241.000000
9,d7fcbd84-8f55-4d9c-bc0d-c4eccf8d5a18,7.48,NaN,170.000000,3041.000000,5912.000000,900.000000,81.0,114.000000,529.000000,24.000000,693.000000,...,3.000000,330.000000,167.000000,4.00,7.000000,1721.000000,60.000000,8098.000000,1840.000000,100.000000
90,cb2c9252-268b-49cd-bdf3-c65d43a4c6bf,7.34,NaN,463.500000,2184.500000,5055.500000,5868.500000,1730.0,1039.000000,554.000000,299.000000,968.000000,...,31.500000,530.500000,367.500000,7451.00,5.500000,1274.500000,44.500000,2.000000,4508.000000,1018.000000
91,bf672818-cfc3-47ab-9c4e-87112164c1fa,7.34,53.0,757.000000,2478.000000,4199.000000,10837.000000,3379.0,1964.000000,579.000000,574.000000,2826.000000,...,60.000000,731.000000,7814.500000,14898.00,4.000000,828.000000,29.000000,65759.000000,7176.000000,1936.000000
92,ef501551-3e8e-4804-973c-5a18c2600e00,7.34,213.0,2000.000000,5497.500000,8995.000000,108.000000,2.0,58.000000,114.000000,6999.000000,4028.666667,...,235.500000,597.500000,676.500000,7760.00,6.000000,94.000000,8.000000,3155.000000,2388.000000,61.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,1f1bf7e1-19d9-48ad-b63d-060be3793ccb,3.0.0.0,48.4,356.111111,484.666667,4289.333333,602.111111,1468.0,892.666667,126.555556,798.888889,1761.888889,...,46.722222,66.222222,1208.888889,90.00,20.444444,282.444444,15.666667,39356.777778,1198.666667,146.222222
57,c7f1a901-d801-4278-bd33-edd8f2126410,3.0.0.0,48.2,233.555556,388.333333,4710.666667,676.555556,1573.0,993.333333,115.777778,933.444444,1896.444444,...,42.611111,62.111111,1381.444444,52.00,23.222222,322.222222,17.333333,43438.888889,1158.333333,105.111111
58,bc9a115a-e90f-4fe4-a5ac-1778130d90b1,3.0,48.0,111.000000,292.000000,5132.000000,751.000000,1678.0,1094.000000,105.000000,1068.000000,2031.000000,...,38.500000,58.000000,1554.000000,14.00,26.000000,362.000000,19.000000,47521.000000,1118.000000,64.000000


In [25]:
df_compare = df_info.join(df_info_interpolated, rsuffix='_int')
df_compare = df_compare.reindex(sorted(df_compare.columns), axis=1)
df_compare

,,,Event.CalendarEntry,Event.CalendarEntry_int,Event.Call,Event.Call_int,Event.CarvedString,Event.CarvedString_int,Event.Chat,Event.Chat_int,Event.Contact,Event.Contact_int,...,File.Audio,File.Audio_int,File.Document,File.Document_int,File.Image,File.Image_int,File.Text,File.Text_int,File.Video,File.Video_int
ufdr,report_id,report_version,,,,,,,,,,,,,,,,,,,,,
89,466ed3f8-526f-4d6f-8b6c-d0950c6baea1,7.34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,27986.0,27986.000000,NaN,14613.500000,1241.0,1241.000000
9,d7fcbd84-8f55-4d9c-bc0d-c4eccf8d5a18,7.48,NaN,NaN,170.0,170.000000,NaN,3041.000000,5912.0,5912.000000,900.0,900.000000,...,1721.0,1721.000000,60.0,60.000000,8098.0,8098.000000,1840.0,1840.000000,100.0,100.000000
90,cb2c9252-268b-49cd-bdf3-c65d43a4c6bf,7.34,NaN,NaN,NaN,463.500000,NaN,2184.500000,NaN,5055.500000,NaN,5868.500000,...,NaN,1274.500000,NaN,44.500000,2.0,2.000000,NaN,4508.000000,NaN,1018.000000
91,bf672818-cfc3-47ab-9c4e-87112164c1fa,7.34,53.0,53.0,757.0,757.000000,NaN,2478.000000,4199.0,4199.000000,10837.0,10837.000000,...,828.0,828.000000,29.0,29.000000,65759.0,65759.000000,7176.0,7176.000000,1936.0,1936.000000
92,ef501551-3e8e-4804-973c-5a18c2600e00,7.34,213.0,213.0,2000.0,2000.000000,NaN,5497.500000,NaN,8995.000000,108.0,108.000000,...,94.0,94.000000,8.0,8.000000,3155.0,3155.000000,2388.0,2388.000000,61.0,61.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,1f1bf7e1-19d9-48ad-b63d-060be3793ccb,3.0.0.0,NaN,48.4,NaN,356.111111,NaN,484.666667,NaN,4289.333333,NaN,602.111111,...,NaN,282.444444,NaN,15.666667,NaN,39356.777778,NaN,1198.666667,NaN,146.222222
57,c7f1a901-d801-4278-bd33-edd8f2126410,3.0.0.0,NaN,48.2,NaN,233.555556,NaN,388.333333,NaN,4710.666667,NaN,676.555556,...,NaN,322.222222,NaN,17.333333,NaN,43438.888889,NaN,1158.333333,NaN,105.111111
58,bc9a115a-e90f-4fe4-a5ac-1778130d90b1,3.0,48.0,48.0,111.0,111.000000,292.0,292.000000,5132.0,5132.000000,751.0,751.000000,...,362.0,362.000000,19.0,19.000000,47521.0,47521.000000,1118.0,1118.000000,64.0,64.000000


In [26]:
df_info_interpolated.mean(axis=0)

Event.CalendarEntry             315.156522
Event.Call                      875.458405
Event.CarvedString             9177.393963
Event.Chat                    22442.427350
Event.Contact                  2147.649573
Event.Cookie                   1379.340503
Event.Email                    1929.330627
Event.InstalledApplication      298.079820
Event.InstantMessage            357.481217
Event.Journey                   753.801730
Event.Location                 1697.330009
Event.Note                      426.946273
Event.Notification              872.941116
Event.Password                 1575.550783
Event.SMS                      5045.858025
Event.VisitedPage              3375.169735
Event.WebBookmark               741.737450
File.Audio                     2228.083523
File.Document                  1387.057217
File.Image                    16258.262712
File.Text                      2437.656780
File.Video                      645.453390
dtype: float64

In [27]:
df_info_interpolated.sum(axis=1)

ufdr  report_id                             report_version
89    466ed3f8-526f-4d6f-8b6c-d0950c6baea1  7.34               43840.500000
9     d7fcbd84-8f55-4d9c-bc0d-c4eccf8d5a18  7.48               25159.000000
90    cb2c9252-268b-49cd-bdf3-c65d43a4c6bf  7.34               36620.000000
91    bf672818-cfc3-47ab-9c4e-87112164c1fa  7.34              131966.500000
92    ef501551-3e8e-4804-973c-5a18c2600e00  7.34               45723.166667
                                                                  ...      
56    1f1bf7e1-19d9-48ad-b63d-060be3793ccb  3.0.0.0            56271.529630
57    c7f1a901-d801-4278-bd33-edd8f2126410  3.0.0.0            61184.514815
58    bc9a115a-e90f-4fe4-a5ac-1778130d90b1  3.0                66097.500000
59    f7ea4441-0112-4e7b-a7e8-6204c9e86cb8  3.0.0.0            97701.000000
6     ff125e4d-ff58-4b25-bb4e-e2de623bd133  5.6               130430.000000
Length: 118, dtype: float64

In [28]:
df_norm_ratios = df_info_interpolated.div(df_info_interpolated.sum(axis=1), axis=0)
df_norm_ratios

,,,Event.CalendarEntry,Event.Call,Event.CarvedString,Event.Chat,Event.Contact,Event.Cookie,Event.Email,Event.InstalledApplication,Event.InstantMessage,Event.Journey,...,Event.Notification,Event.Password,Event.SMS,Event.VisitedPage,Event.WebBookmark,File.Audio,File.Document,File.Image,File.Text,File.Video
ufdr,report_id,report_version,,,,,,,,,,,,,,,,,,,,,
89,466ed3f8-526f-4d6f-8b6c-d0950c6baea1,7.34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.638360,0.333333,0.028307
9,d7fcbd84-8f55-4d9c-bc0d-c4eccf8d5a18,7.48,NaN,0.006757,0.120871,0.234985,0.035772,0.003220,0.004531,0.021026,0.000954,0.027545,...,0.000119,0.013117,0.006638,0.000159,0.000278,0.068405,0.002385,0.321873,0.073135,0.003975
90,cb2c9252-268b-49cd-bdf3-c65d43a4c6bf,7.34,NaN,0.012657,0.059653,0.138053,0.160254,0.047242,0.028372,0.015128,0.008165,0.026434,...,0.000860,0.014487,0.010035,0.203468,0.000150,0.034803,0.001215,0.000055,0.123102,0.027799
91,bf672818-cfc3-47ab-9c4e-87112164c1fa,7.34,0.000402,0.005736,0.018777,0.031819,0.082119,0.025605,0.014883,0.004387,0.004350,0.021415,...,0.000455,0.005539,0.059216,0.112892,0.000030,0.006274,0.000220,0.498301,0.054377,0.014670
92,ef501551-3e8e-4804-973c-5a18c2600e00,7.34,0.004658,0.043742,0.120234,0.196727,0.002362,0.000044,0.001269,0.002493,0.153073,0.088110,...,0.005151,0.013068,0.014796,0.169717,0.000131,0.002056,0.000175,0.069002,0.052227,0.001334
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,1f1bf7e1-19d9-48ad-b63d-060be3793ccb,3.0.0.0,0.000860,0.006328,0.008613,0.076226,0.010700,0.026088,0.015864,0.002249,0.014197,0.031310,...,0.000830,0.001177,0.021483,0.001599,0.000363,0.005019,0.000278,0.699408,0.021301,0.002599
57,c7f1a901-d801-4278-bd33-edd8f2126410,3.0.0.0,0.000788,0.003817,0.006347,0.076991,0.011058,0.025709,0.016235,0.001892,0.015256,0.030995,...,0.000696,0.001015,0.022578,0.000850,0.000380,0.005266,0.000283,0.709965,0.018932,0.001718
58,bc9a115a-e90f-4fe4-a5ac-1778130d90b1,3.0,0.000726,0.001679,0.004418,0.077643,0.011362,0.025387,0.016551,0.001589,0.016158,0.030727,...,0.000582,0.000877,0.023511,0.000212,0.000393,0.005477,0.000287,0.718953,0.016914,0.000968


In [29]:
#Example
np.random.multinomial(1000, df_norm_ratios.mean(axis=0).values)

array([  5,  17,  88, 253,  25,  28,  28,   3,  12,   8,  30,  11,  14,
        22,  67,  53,  17,  43,  26, 200,  38,  12])

# Final Event Distribution

In [30]:
df_norm_ratios.mean(axis=0)

Event.CalendarEntry           0.008084
Event.Call                    0.018017
Event.CarvedString            0.088888
Event.Chat                    0.233172
Event.Contact                 0.031199
Event.Cookie                  0.029607
Event.Email                   0.028409
Event.InstalledApplication    0.008118
Event.InstantMessage          0.007496
Event.Journey                 0.013641
Event.Location                0.029488
Event.Note                    0.008700
Event.Notification            0.013001
Event.Password                0.021260
Event.SMS                     0.068718
Event.VisitedPage             0.052968
Event.WebBookmark             0.015689
File.Audio                    0.027114
File.Document                 0.023558
File.Image                    0.226595
File.Text                     0.036716
File.Video                    0.015860
dtype: float64

In [33]:
final_event_distribution = df_norm_ratios.mean(axis=0) / df_norm_ratios.mean(axis=0).sum()
final_event_distribution

Event.CalendarEntry           0.008034
Event.Call                    0.017905
Event.CarvedString            0.088331
Event.Chat                    0.231712
Event.Contact                 0.031004
Event.Cookie                  0.029422
Event.Email                   0.028231
Event.InstalledApplication    0.008067
Event.InstantMessage          0.007449
Event.Journey                 0.013555
Event.Location                0.029304
Event.Note                    0.008645
Event.Notification            0.012920
Event.Password                0.021127
Event.SMS                     0.068287
Event.VisitedPage             0.052637
Event.WebBookmark             0.015591
File.Audio                    0.026944
File.Document                 0.023411
File.Image                    0.225176
File.Text                     0.036486
File.Video                    0.015761
dtype: float64

In [37]:
for event_type, weight in final_event_distribution.iteritems():
    print(event_type, weight)

Event.CalendarEntry 0.008033778059093458
Event.Call 0.017904579120129726
Event.CarvedString 0.08833110186977366
Event.Chat 0.23171214725154632
Event.Contact 0.031003714382103302
Event.Cookie 0.029421772444486495
Event.Email 0.028230869996553703
Event.InstalledApplication 0.008067357450615154
Event.InstantMessage 0.0074493023553380165
Event.Journey 0.013555383125729948
Event.Location 0.029303628736092973
Event.Note 0.008645250000578492
Event.Notification 0.0129200222900151
Event.Password 0.021126872324414167
Event.SMS 0.06828742525523863
Event.VisitedPage 0.0526368209726625
Event.WebBookmark 0.015591234043966382
File.Audio 0.026944272773524385
File.Document 0.023410996308425542
File.Image 0.22517613225398656
File.Text 0.03648646604713487
File.Video 0.015760872938590718


In [38]:
0.06828742525523863 + 0.0074493023553380165

0.07573672761057665

In [39]:
0.03648646604713487 + 0.023410996308425542

0.05989746235556041

# File Distribution

In [10]:
FILE_TYPES = {'Image','Video','Audio','Document','Text'}

In [11]:
df_file_info = df_info[[f'File.{ft}' for ft in FILE_TYPES]]
df_file_info

,,,File.Image,File.Audio,File.Text,File.Document,File.Video
ufdr,report_id,report_version,,,,,
89,466ed3f8-526f-4d6f-8b6c-d0950c6baea1,7.34,27986.0,NaN,NaN,NaN,1241.0
9,d7fcbd84-8f55-4d9c-bc0d-c4eccf8d5a18,7.48,8098.0,1721.0,1840.0,60.0,100.0
90,cb2c9252-268b-49cd-bdf3-c65d43a4c6bf,7.34,2.0,NaN,NaN,NaN,NaN
91,bf672818-cfc3-47ab-9c4e-87112164c1fa,7.34,65759.0,828.0,7176.0,29.0,1936.0
92,ef501551-3e8e-4804-973c-5a18c2600e00,7.34,3155.0,94.0,2388.0,8.0,61.0
...,...,...,...,...,...,...,...
56,1f1bf7e1-19d9-48ad-b63d-060be3793ccb,3.0.0.0,NaN,NaN,NaN,NaN,NaN
57,c7f1a901-d801-4278-bd33-edd8f2126410,3.0.0.0,NaN,NaN,NaN,NaN,NaN
58,bc9a115a-e90f-4fe4-a5ac-1778130d90b1,3.0,47521.0,362.0,1118.0,19.0,64.0


In [12]:
def get_file_sizes(info_path):
    with info_path.open() as f:
        info_json = json.load(f)
    file_sizes = [
        {'ufdr': info_path.name[4:-5], 'report_version': info_json.get('report_version'),
         'category': file_info["category"], 'size': file_info["size"]}
        for file_info in info_json['file_infos']
        if file_info["category"] in FILE_TYPES
    ]
    return file_sizes

In [13]:
df_file_sizes = pd.DataFrame((row for info_path in info_dirpath.iterdir() for row in get_file_sizes(info_path)))\
    .set_index(['ufdr', 'report_version'])\
    .sort_values('report_version', ascending=False)
df_file_sizes = df_file_sizes.reindex(sorted(df_file_sizes.columns), axis=1)
df_file_sizes

category   size
ufdr report_version                
8    7.48              Audio  73174
106  7.48              Image   5194
     7.48              Image   1768
     7.48              Image   6071
     7.48              Image   4943
...                      ...    ...
60   3.0               Image   2960
     3.0               Image   1648
     3.0               Image   1707
     3.0               Image   2414
58   3.0               Image   1102

[1588640 rows x 2 columns]

In [44]:
df_file_sizes.to_csv('/home/danielh/dev/raw_file_stats_real_ufdrs.csv')

In [14]:
df_file_sizes.groupby('category').describe()

size                                                       \
              count          mean           std  min       25%       50%   
category                                                                   
Audio      158409.0  3.759094e+05  2.038159e+06  0.0   5231.00   27265.0   
Document     3254.0  1.031431e+06  6.469269e+06  0.0   5671.00   39156.0   
Image     1244925.0  9.760085e+04  5.402960e+05  0.0   2572.00    8258.0   
Text       136960.0  1.459077e+04  2.002032e+05  0.0    503.00    1869.0   
Video       45092.0  4.477113e+06  2.874294e+07  0.0  33672.25  240211.5   

                                    
                 75%           max  
category                            
Audio      136161.00  1.577639e+08  
Document   182196.75  1.589103e+08  
Image       35035.00  3.187435e+07  
Text         4317.00  2.762093e+07  
Video     1533675.00  1.686948e+09

Find UFDRs which don't look realistic and exclude them from consideration

In [43]:
df_file_size_total = df_file_sizes.groupby(['ufdr','category']).size().to_frame('size')
df_file_size_total = df_file_size_total.reset_index(level='category').pivot(columns='category', values='size')
df_file_size_total['total'] = df_file_size_total.sum(axis=1)
df_file_size_total = df_file_size_total.sort_values('total')
df_file_size_total

category,Audio,Document,Image,Text,Video,total
ufdr,,,,,,
98,NaN,NaN,1.0,NaN,NaN,1.0
19,NaN,NaN,1.0,NaN,NaN,1.0
13,NaN,NaN,1.0,NaN,NaN,1.0
90,NaN,NaN,2.0,NaN,NaN,2.0
10,NaN,NaN,3.0,NaN,NaN,3.0
...,...,...,...,...,...,...
100,22668.0,151.0,48560.0,18037.0,4054.0,93470.0
101,22668.0,151.0,48560.0,18037.0,4054.0,93470.0
114,32349.0,394.0,51572.0,3672.0,8674.0,96661.0


In [ ]:
df_file_sizes[df_file_sizes.]